In [ ]:
from typing import TypedDict, Literal, Sequence

from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain.messages import HumanMessage
from dotenv import load_dotenv
from rich import print

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)


#1. 状態を定義
class OverAllState(TypedDict):
    topic: str
    content_type: Literal["poem", "joke"]
    content_Chinese: str
    poem: str
    joke: str


#2. ルーターノードを定義
def router(state: OverAllState) -> Command[Literal["poem_node", "joke_node", "__end__"]]:
    my_state = state["content_type"]

    if my_state == "poem":
        return Command(
            update={
                "content_Chinese": "一つの詩",
        },
        goto="poem_node"
        )
    elif my_state == "joke":
        return Command(
            update={
                "content_Chinese": "一つのジョーク",
            },
            goto="joke_node"
        )
    else:
        return Command(
            update={
                "content_Chinese": "error",
            },
            goto=END
        )


def poem_node(state: OverAllState) -> OverAllState:
    poem = model.invoke([f"{state['topic']}をテーマにした{state['content_Chinese']}を書いてください"]).content
    return {
        
        "poem": poem
    }


def joke_node(state: OverAllState) -> OverAllState:
    joke = model.invoke([f"{state['topic']}をテーマにした{state['content_Chinese']}を書いてください"]).content
    return {
        "joke": joke
    }


# 3. グラフを構築
builder = StateGraph(state_schema=OverAllState)
builder.add_node("router", router)
builder.add_node("poem_node", poem_node)
builder.add_node("joke_node", joke_node)

builder.add_edge(START, "router")
builder.add_edge("poem_node", END)
builder.add_edge("joke_node", END)

graph = builder.compile()

res1 = graph.invoke({"topic": "桜", "content_type": "poem"})
print(res1)
print("=" * 50)
res2 = graph.invoke({"topic": "猫", "content_type": "joke"})
print(res2)
res3 = graph.invoke({"topic": "猫", "content_type": "xxx"})
print(res3)

from IPython.display import display

display(graph)

